# Day 3 | 분기 EPS 수집 + YoY 성장률 계산
## Korean Stock Undervaluation Analysis — Quarterly EPS & YoY

**목표:**
- DART 분기보고서에서 분기 EPS 수집 (133개 종목)
- TTM(최근 12개월) 기준 YoY 성장률 산출
- 괴리율 v2 계산 (고평가 구간 YoY 반영 완화)

**핵심 트러블슈팅:**
- corp_code CSV 저장 시 앞자리 '00' 패딩 소실 → `str.zfill(8)` 필수
- EPS 계정명이 기업마다 상이 (20여 가지 변형 존재)
- 고성장 기업 YoY 이상치 (1426%) → 100% 캡핑 필요

**데이터 흐름:**
```
clean_data_final.csv (133개)
    ↓
DART 분기보고서 API (분기 EPS × 8분기)
    ↓
TTM EPS → YoY 성장률
    ↓
gap_v1 + YoY → gap_v2 (조정괴리율)
    ↓
gap_v2.csv (136개)
```


## 0. 환경 세팅

In [ ]:
!pip install -q git+https://github.com/FinanceData/FinanceDataReader.git

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import requests
import pickle
import time
import os
from datetime import datetime

API_KEY   = "여기에_DART_API_키_입력"
BASE_PATH = '/content/drive/MyDrive/stock-analysis'

# ⚠️ corp_code 반드시 str + zfill(8) 적용
# CSV 저장 시 숫자형으로 읽혀 앞자리 00이 소실되는 문제
df_main = pd.read_csv(
    f'{BASE_PATH}/data/processed/clean_data_final.csv',
    dtype={'Code': str, 'corp_code': str}
)
df_main['corp_code'] = df_main['corp_code'].str.zfill(8)

df_v1 = pd.read_csv(
    f'{BASE_PATH}/data/output/gap_v1_refined.csv',
    dtype={'Code': str, 'corp_code': str}
)

print(f"✅ df_main: {len(df_main)}개")
print(f"✅ df_v1  : {len(df_v1)}개")

# 패딩 검증
assert df_main['corp_code'].str.len().eq(8).all(), "❌ corp_code 길이 오류!"
print("✅ corp_code 패딩 검증 완료")


## 1. 분기 EPS 수집 함수

In [ ]:
# DART 기업마다 EPS 계정명이 다름 → 20여 가지 변형 대응
EPS_ACCOUNT_NAMES = {
    '기본주당이익',
    '기본주당순이익',
    '기본주당순이익(손실)',
    '기본 주당순이익(손실)',
    '주당순이익',
    '주당이익',
    '기본주당이익(손실)',
    '주당손익',
    '기본주당손익',
    '보통주기본주당이익(손실)',
    '보통주기본주당이익',
    '보통주 기본주당순이익',
    '보통주기본주당순이익(손실)',
    '보통주 기본및희석주당이익',
    '보통주 기본주당이익',
    '1. 기본주당이익',
    '기본주당보통주순이익(손실)',
    '기본주당이익(손실) 합계',
    '기본주당이익(단위: 원)',
    '지배기업 소유주 기본주당이익',
    '계속영업기본주당이익(손실)',  # 중단영업 제외, 계속영업 우선
}

def get_quarterly_eps(corp_code: str, api_key: str) -> pd.DataFrame:
    """
    DART 분기보고서 EPS 수집
    - CFS(연결) 우선, 없으면 OFS(별도) fallback
    - 계속영업 EPS 우선 (중단영업 일회성 손익 제외)
    - 반환: DataFrame(year, quarter, reprt_code, eps)
    """
    url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
    results = []
    current_year = datetime.now().year
    qtr_label = {'11013': 'Q1', '11012': 'Q2', '11014': 'Q3', '11011': 'Q4'}

    targets = []
    for year in [current_year - 1, current_year - 2]:
        for reprt_code in ['11011', '11014', '11012', '11013']:
            targets.append((str(year), reprt_code))

    for year, reprt_code in targets:
        eps = None
        for fs_div in ['CFS', 'OFS']:
            params = {
                'crtfc_key': api_key,
                'corp_code': corp_code,
                'bsns_year': year,
                'reprt_code': reprt_code,
                'fs_div': fs_div,
            }
            try:
                r    = requests.get(url, params=params, timeout=10)
                data = r.json()
                if data.get('status') != '000':
                    continue
                for item in data.get('list', []):
                    if item.get('account_nm', '').strip() in EPS_ACCOUNT_NAMES:
                        raw = item.get('thstrm_amount', '').replace(',', '').strip()
                        try:
                            eps = float(raw)
                        except ValueError:
                            pass
                        break
                if eps is not None:
                    break
            except Exception:
                pass
            time.sleep(0.15)

        if eps is not None:
            results.append({
                'year':       int(year),
                'quarter':    qtr_label[reprt_code],
                'reprt_code': reprt_code,
                'eps':        eps
            })

    return pd.DataFrame(results)


## 2. 전체 133개 분기 EPS 수집 (약 15~20분)

In [ ]:
# pickle 체크포인트: 세션 만료 대비 20개마다 자동 저장
os.makedirs(f'{BASE_PATH}/data/temp', exist_ok=True)

all_eps = {}
failed  = []
total   = len(df_main)

print(f"🚀 분기 EPS 수집 시작: {total}개")
print("=" * 50)

for idx, (_, row) in enumerate(df_main.iterrows(), 1):
    corp_code = row['corp_code']
    name      = row['Name']
    code      = row['Code']

    eps_df = get_quarterly_eps(corp_code, API_KEY)

    if len(eps_df) > 0:
        all_eps[code] = eps_df
    else:
        failed.append({'Code': code, 'Name': name})

    # 20개마다 진행상황 출력 + 체크포인트 저장
    if idx % 20 == 0 or idx == total:
        print(f"  [{idx:3d}/{total}] 성공: {len(all_eps)}개 | 실패: {len(failed)}개")
        with open(f'{BASE_PATH}/data/temp/all_eps_checkpoint.pkl', 'wb') as f:
            pickle.dump(all_eps, f)
        print(f"  💾 체크포인트 저장 완료")

print(f"\n✅ 수집 완료: {len(all_eps)}개 / 실패: {len(failed)}개")
if failed:
    print("\n❌ 실패 종목:")
    for f in failed:
        print(f"  {f['Name']} ({f['Code']})")


## 3. YoY 성장률 계산 (TTM 기준)

In [ ]:
def calc_yoy_growth(eps_dict: dict) -> pd.DataFrame:
    """
    TTM(최근 12개월) EPS 기준 YoY 성장률 계산
    - TTM_recent: 최근 4분기 EPS 합산
    - TTM_prev  : 1년 전 4분기 EPS 합산
    - YoY       : (TTM_recent - TTM_prev) / |TTM_prev| × 100
    - 8분기 미만 수집 시 YoY = None
    """
    rows = []
    qtr_order = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}

    for code, df in eps_dict.items():
        if df.empty or len(df) < 4:
            continue

        df = df.copy()
        df['qtr_num'] = df['quarter'].map(qtr_order)
        df = (df.sort_values(['year', 'qtr_num'])
                .drop_duplicates(subset=['year', 'quarter'], keep='last')
                .reset_index(drop=True))

        n = len(df)

        if n >= 8:
            ttm_recent = df.tail(4)['eps'].sum()
            ttm_prev   = df.iloc[-8:-4]['eps'].sum()
            yoy = (ttm_recent - ttm_prev) / abs(ttm_prev) * 100 if ttm_prev != 0 else None
        elif n >= 4:
            ttm_recent = df.tail(4)['eps'].sum()
            ttm_prev   = None
            yoy        = None
        else:
            continue

        rows.append({
            'Code':           code,
            'ttm_eps_recent': round(ttm_recent, 0),
            'ttm_eps_prev':   round(ttm_prev, 0) if ttm_prev is not None else None,
            'yoy_growth_pct': round(yoy, 1) if yoy is not None else None,
            'quarters_used':  n,
        })

    return pd.DataFrame(rows)


df_yoy = calc_yoy_growth(all_eps)
print(f"✅ YoY 계산 완료: {len(df_yoy)}개")
print(f"   YoY 있음: {df_yoy['yoy_growth_pct'].notna().sum()}개")
print(f"   YoY 없음: {df_yoy['yoy_growth_pct'].isna().sum()}개 (분기 부족)")


## 4. 괴리율 v2 산출 (YoY 성장률 반영)

In [ ]:
# v1 + YoY 병합
df_v2 = df_v1.merge(df_yoy, on='Code', how='left')

# 조정괴리율 공식:
# - 고평가 구간(gap > 0): 성장률만큼 완화 → gap × (1 - yoy_decimal × 0.5)
# - 저평가 구간(gap < 0): 유지 (성장률로 저평가를 더 확대하지 않음)
# - YoY 상한: 100% 캡핑 (1426% 같은 턴어라운드 이상치 방지)
ADJUST_FACTOR = 0.5

def apply_growth_adjustment(gap, yoy_pct):
    """
    YoY 성장률 기반 괴리율 조정
    - 고평가(gap > 0)만 완화, 저평가(gap < 0)는 유지
    - YoY 상한 100% 캡핑
    """
    if pd.isna(gap):
        return None
    if pd.isna(yoy_pct) or yoy_pct <= 0:
        return round(gap, 4)
    yoy_decimal = min(yoy_pct, 100.0) / 100.0
    if gap > 0:
        adjusted = gap - (gap * yoy_decimal * ADJUST_FACTOR)
    else:
        adjusted = gap
    return round(adjusted, 4)

df_v2['gap_market_v2'] = df_v2.apply(
    lambda r: apply_growth_adjustment(r['gap_market'], r['yoy_growth_pct']), axis=1)
df_v2['gap_sector_v2'] = df_v2.apply(
    lambda r: apply_growth_adjustment(r['gap_sector'], r['yoy_growth_pct']), axis=1)
df_v2['gap_market_delta'] = (df_v2['gap_market_v2'] - df_v2['gap_market']).round(4)
df_v2['gap_sector_delta'] = (df_v2['gap_sector_v2'] - df_v2['gap_sector']).round(4)

# signal_v2
def make_signal_v2(row):
    gm = row['gap_market_v2']
    gs = row['gap_sector_v2']
    if pd.isna(gm) or pd.isna(gs):
        return 'UNKNOWN'
    avg = (gm + gs) / 2
    if avg <= -0.30:   return '강력매수'
    elif avg <= -0.15: return '매수'
    elif avg >= 0.30:  return '강력매도'
    elif avg >= 0.15:  return '매도'
    else:              return '중립'

df_v2['signal_v2'] = df_v2.apply(make_signal_v2, axis=1)

# 저장
df_v2.to_csv(f'{BASE_PATH}/data/output/gap_v2.csv', index=False, encoding='utf-8-sig')
print(f"✅ gap_v2.csv 저장: {len(df_v2)}개")
print("\nsignal_v2 분포:")
print(df_v2['signal_v2'].value_counts())
